# STL Playground: LOESS und robuste LOESS verstehen

Dieses Notebook ist ein **Playground** zum Experimentieren mit einem zentralen Baustein der STL-Methode: LOESS. STL zerlegt eine Zeitreihe später in Trend, Saisonalität und Residuen. Bevor diese Zerlegung sinnvoll interpretiert werden kann, ist es wichtig zu verstehen, wie lokale Glättung funktioniert.

Der Fokus liegt deshalb bewusst nicht auf einem realen Business-Datensatz, sondern auf einer kontrollierten Sinuskurve mit Rauschen. Dadurch sehen wir klarer, wie LOESS reagiert, wenn das Glättungsfenster klein, mittel oder groß ist. Danach betrachten wir die lokalen Gewichte und erweitern das Beispiel um Ausreißer, um robuste LOESS zu verstehen.

Ziel des Notebooks ist es, einzelne Mechanismen hinter STL sichtbar zu machen:

- Wie funktioniert lokale Regression mit gewichteten Nachbarschaftspunkten?
- Wie verändert die Fenstergröße die Glättung?
- Wie entstehen Distanzgewichte in LOESS?
- Wie reduziert robuste LOESS den Einfluss von Ausreißern?
- Warum sind diese Ideen wichtig, wenn STL später Trend, Saisonalität und Residuen schätzt?

In [ ]:
# Import core libraries for numerical work, visualization, and LOESS smoothing.
import warnings

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")

# Set a consistent visual style for all plots.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


## 1. LOESS-Intuition

LOESS steht für **locally estimated scatterplot smoothing**. Die Grundidee ist, nicht eine einzige globale Kurve an alle Datenpunkte anzupassen, sondern für jeden Zielpunkt eine lokale Regression zu berechnen. Diese lokale Regression nutzt vor allem Punkte in der Nähe des Zielpunkts.

In diesem Playground verwenden wir eine künstliche Sinuskurve mit Rauschen. Die wahre Kurve wird nur zur Erzeugung der Daten genutzt. In den Plots sehen wir bewusst nur die verrauschten Beobachtungen und die LOESS-Glättung, damit der Blick auf das Verhalten des Glättungsverfahrens gerichtet bleibt.

Wir vergleichen drei Fenstergrößen:

- Ein kleines Fenster reagiert sehr flexibel, kann aber Rauschen mitlernen.
- Ein mittleres Fenster ist oft ein guter Kompromiss.
- Ein großes Fenster erzeugt eine ruhigere, aber weniger flexible Kurve.

In [ ]:
# Create a small noisy example to demonstrate LOESS smoothing.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120)
y_true = np.sin(x) + 0.15 * x
y_noisy = y_true + rng.normal(0, 0.35, size=len(x))

# Fit LOESS curves with different spans.
loess_small = lowess(y_noisy, x, frac=0.08, return_sorted=False)
loess_medium = lowess(y_noisy, x, frac=0.22, return_sorted=False)
loess_large = lowess(y_noisy, x, frac=0.50, return_sorted=False)

# Compare the smoothing windows in separate vertical subplots.
loess_settings = [
    ("Small span: flexible", loess_small, "#d95f02"),
    ("Medium span: balanced", loess_medium, "#1b9e77"),
    ("Large span: smooth", loess_large, "#7570b3"),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, sharey=True)

for ax, (title, loess_curve, color) in zip(axes, loess_settings):
    ax.scatter(x, y_noisy, s=20, alpha=0.40, color="#4b5563", label="Noisy observations")
    ax.plot(x, loess_curve, color=color, linewidth=2.5, label="LOESS smooth")
    ax.set_title(title)
    ax.set_ylabel("y")
    ax.legend(loc="upper left")

axes[-1].set_xlabel("x")
fig.suptitle("LOESS: Vergleich kleiner, mittlerer und großer Glättungsfenster", fontsize=16)
plt.tight_layout()
plt.show()


## 2. Lokale Regression, Nachbarschaftsfenster und gewichtete Punkte

LOESS besteht aus drei zentralen Bausteinen.

**Lokale Regression:** Für jeden Punkt wird nicht die gesamte Zeitreihe gleich stark verwendet. Stattdessen wird in der Nähe des Zielpunkts eine kleine Regression berechnet. Meist ist diese Regression linear oder quadratisch.

**Nachbarschaftsfenster:** Das Fenster bestimmt, wie viele benachbarte Punkte in die lokale Regression eingehen. Ein kleines Fenster reagiert stark auf lokale Bewegungen. Ein großes Fenster glättet stärker und ignoriert kurzfristige Schwankungen eher.

**Gewichtete Punkte:** Innerhalb des Fensters sind nicht alle Punkte gleich wichtig. Punkte nahe am Zielpunkt erhalten höhere Gewichte. Punkte weiter entfernt erhalten geringere Gewichte. Typisch ist eine Gewichtsfunktion, die am Rand des Fensters gegen null geht.

Der wichtigste Parameter ist deshalb die Fenstergröße beziehungsweise der **Span**. In `statsmodels.lowess` wird dieser als `frac` angegeben. Ein Wert von `0.20` bedeutet: Für jede lokale Regression werden ungefähr 20 Prozent der Datenpunkte als Nachbarschaft verwendet.

In [ ]:
# Visualize the neighborhood and weights for one LOESS target point.
target_x = 5.0
frac = 0.22
n_neighbors = int(np.ceil(frac * len(x)))
distances = np.abs(x - target_x)
max_distance = np.partition(distances, n_neighbors)[n_neighbors]

# Compute tricube weights used conceptually by LOESS.
weights = np.where(
    distances <= max_distance,
    (1 - (distances / max_distance) ** 3) ** 3,
    0,
)

fig, ax = plt.subplots(figsize=(14, 5))
scatter = ax.scatter(x, y_noisy, c=weights, cmap="viridis", s=45, edgecolor="white", linewidth=0.4)
ax.axvline(target_x, color="#b91c1c", linestyle="--", linewidth=2, label="Target point")
ax.set_title("LOESS-Nachbarschaft: nahe Punkte erhalten höhere Gewichte")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Local weight")
plt.show()

## 3. Robuste LOESS-Gewichte

Die bisherige LOESS-Erklärung betrachtet vor allem die **Distanzgewichte**: Punkte nahe am Zielpunkt erhalten hohe Gewichte, entfernte Punkte niedrige Gewichte. Robuste LOESS erweitert diese Idee um eine zweite Gewichtung. Nach einer ersten lokalen Glättung werden die Residuen betrachtet. Beobachtungen mit sehr großen Residuen werden als potenzielle Ausreißer behandelt und erhalten ein kleineres robustes Gewicht.

Dadurch entsteht konzeptionell ein kombiniertes Gewicht:

\[
w_{gesamt} = w_{Distanz} \cdot w_{Robust}
\]

Ein Punkt kann also sehr nahe am Zielpunkt liegen und trotzdem wenig Einfluss haben, wenn sein Residuum extrem groß ist. Genau das macht robuste LOESS nützlich, wenn einzelne Ausreißer die lokale Regression verzerren würden.

Im folgenden Beispiel verwenden wir dieselbe synthetische Reihe wie zuvor, fügen aber drei künstliche Ausreißer hinzu. Danach vergleichen wir Standard-LOESS mit robuster LOESS und visualisieren, wie sich die Gewichte verändern.

In [ ]:
# Study robust LOESS on the same noisy example with artificial outliers.
y_noisy_outliers = y_noisy.copy()
outlier_indices = [24, 61, 92]
outlier_impacts = [2.3, -2.6, 2.1]

for index, impact in zip(outlier_indices, outlier_impacts):
    y_noisy_outliers[index] += impact

# Fit standard and robust LOESS curves.
standard_loess = lowess(y_noisy_outliers, x, frac=0.22, it=0, return_sorted=False)
robust_loess = lowess(y_noisy_outliers, x, frac=0.22, it=3, return_sorted=False)

# Approximate the robust residual weights used after an initial LOESS fit.
residuals = y_noisy_outliers - standard_loess
mad = np.median(np.abs(residuals - np.median(residuals)))
robust_scale = 6 * mad

robust_weights = np.where(
    np.abs(residuals) < robust_scale,
    (1 - (residuals / robust_scale) ** 2) ** 2,
    0,
)

# Combine distance weights around the same target point with robust residual weights.
combined_weights = weights * robust_weights

fig, axes = plt.subplots(3, 1, figsize=(14, 15), sharex=True, sharey=True)

axes[0].scatter(x, y_noisy_outliers, s=28, alpha=0.45, color="#4b5563", label="Noisy observations")
axes[0].scatter(x[outlier_indices], y_noisy_outliers[outlier_indices], s=90, color="#dc2626", label="Artificial outliers")
axes[0].plot(x, standard_loess, color="#d95f02", linewidth=2.5, label="Standard LOESS")
axes[0].plot(x, robust_loess, color="#059669", linewidth=2.5, label="Robust LOESS")
axes[0].set_title("Standard-LOESS vs. robuste LOESS")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper left")

normal_scatter = axes[1].scatter(
    x,
    y_noisy_outliers,
    c=weights,
    cmap="viridis",
    s=45,
    edgecolor="white",
    linewidth=0.4,
)
axes[1].scatter(x[outlier_indices], y_noisy_outliers[outlier_indices], s=110, facecolors="none", edgecolors="#dc2626", linewidth=2, label="Artificial outliers")
axes[1].plot(x, standard_loess, color="#d95f02", linewidth=2.5, label="Standard LOESS")
axes[1].axvline(target_x, color="#b91c1c", linestyle="--", linewidth=2, label="Target point")
axes[1].set_title("Normale LOESS: Distanzgewichte als Heatmap")
axes[1].set_ylabel("y")
axes[1].legend(loc="upper left")
cbar = plt.colorbar(normal_scatter, ax=axes[1])
cbar.set_label("Distance weight")

robust_scatter = axes[2].scatter(
    x,
    y_noisy_outliers,
    c=combined_weights,
    cmap="viridis",
    s=45,
    edgecolor="white",
    linewidth=0.4,
)
axes[2].scatter(x[outlier_indices], y_noisy_outliers[outlier_indices], s=110, facecolors="none", edgecolors="#dc2626", linewidth=2, label="Artificial outliers")
axes[2].plot(x, robust_loess, color="#059669", linewidth=2.5, label="Robust LOESS")
axes[2].axvline(target_x, color="#b91c1c", linestyle="--", linewidth=2, label="Target point")
axes[2].set_title("Robuste LOESS: kombinierte Gewichte als Heatmap")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")
axes[2].legend(loc="upper left")
cbar = plt.colorbar(robust_scatter, ax=axes[2])
cbar.set_label("Distance weight × robust weight")

plt.tight_layout()
plt.show()


In [ ]:
# Compare distance, robust, and combined weights for the same target point.
weight_panels = [
    ("Distance weights", weights),
    ("Robust residual weights", robust_weights),
    ("Combined weights", combined_weights),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, sharey=True)

for ax, (title, panel_weights) in zip(axes, weight_panels):
    scatter = ax.scatter(
        x,
        y_noisy_outliers,
        c=panel_weights,
        cmap="viridis",
        s=45,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.scatter(x[outlier_indices], y_noisy_outliers[outlier_indices], s=110, facecolors="none", edgecolors="#dc2626", linewidth=2)
    ax.axvline(target_x, color="#b91c1c", linestyle="--", linewidth=2)
    ax.set_title(title)
    ax.set_ylabel("y")
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label("Weight")

axes[-1].set_xlabel("x")
fig.suptitle("LOESS-Gewichte: Distanz, Robustheit und Kombination", fontsize=16)
plt.tight_layout()
plt.show()
